# 02. Prompt Engineering 실습
**SK하이닉스 Autonomous R&D — Day 3** 

---
## 0. 준비

In [2]:
from openai import OpenAI
from dotenv import load_dotenv
import os

load_dotenv()
client = OpenAI(api_key=os.getenv('OPENAI_API_KEY'))


def ask(system_prompt, user_prompt, temperature=0.2):
    response = client.chat.completions.create(
        model='gpt-4o-mini',
        temperature=temperature,
        messages=[
            {'role': 'system', 'content': system_prompt},
            {'role': 'user', 'content': user_prompt},
        ],
    )
    return response.choices[0].message.content

In [ ]:
# 기본적으로 같은 모델이라도 프롬프트에 따라 다른 출력을 생성

system = 'You are a helpful assistant. Answer in Korean.'

prompts_ko = [
    "프랑스의 수도는",
    "Q: 프랑스의 수도는 어디인가요? A:",
    "프랑스의 수도 도시는",
]

for prompt in prompts_ko:
    print(f"프롬프트: {prompt}")
    print(f"생성: {ask(system, prompt, temperature=0.2)}")
    print()

---
## 1. Zero-shot vs Few-shot 

| 방식 | 설명 |
|------|------|
| **Zero-shot** | 예시 없이 지시만 |
| **Few-shot** | 원하는 형식의 **예시**를 함께 제공 |

In [3]:
system = 'You are a helpful assistant. Answer in Korean.'
# Zero-shot — 형식 지시 없음
user_zero = '''
아래 영화 리뷰가 긍정인지 부정인지 판정해줘.
- "연출은 좋았지만 2시간이 너무 길었다"
- "최고의 영화, 또 보고 싶다"
'''
print('=== Zero-shot ===')
print(ask(system, user_zero))

=== Zero-shot ===
첫 번째 리뷰는 부정적인 요소가 포함되어 있어 부정으로 판정할 수 있습니다. 두 번째 리뷰는 긍정적인 표현이므로 긍정으로 판정할 수 있습니다.


In [4]:
# Few-shot — 출력 형식 예시 포함
user_few = '''
아래 형식으로 감성 판정해줘.

[예시]
입력: "배우 연기가 훌륭했다" → 긍정 | 9/10
입력: "스토리가 지루했다" → 부정 | 3/10

[실제 데이터]
- "연출은 좋았지만 2시간이 너무 길었다"
- "최고의 영화, 또 보고 싶다"
'''
print('=== Few-shot ===')
print(ask(system, user_few))

=== Few-shot ===
입력: "연출은 좋았지만 2시간이 너무 길었다" → 긍정 | 6/10  
입력: "최고의 영화, 또 보고 싶다" → 긍정 | 10/10


In [5]:
zero_shot_prompt_ko = """다음은 단어 유추의 예시입니다:

문제: 행복한 : 슬픈 :: 좋은 :"""

print(ask(system, zero_shot_prompt_ko, temperature=0.7))

좋은 : 나쁜


In [6]:
few_shot_prompt_ko = """다음은 단어 유추의 예시입니다:

뜨거운 : 차가운 :: 위 : 아래
큰 : 작은 :: 빠른 : 느린
낮 : 밤 :: 밝은 : 어두운
행복한 : 슬픈 :: 좋은 :"""

print('Few-shot 프롬프트:')
print(few_shot_prompt_ko)
print('\n생성된 완성:')
print(ask(system, few_shot_prompt_ko, temperature=0.7))

Few-shot 프롬프트:
다음은 단어 유추의 예시입니다:

뜨거운 : 차가운 :: 위 : 아래
큰 : 작은 :: 빠른 : 느린
낮 : 밤 :: 밝은 : 어두운
행복한 : 슬픈 :: 좋은 :

생성된 완성:
행복한 : 슬픈 :: 좋은 : 나쁜


---
## 2. 역할(Role) 설정 

In [7]:
question = '하루 커피 4잔 마셔도 괜찮을까?'

print('=== 일반 assistant ===')
print(ask('You are a helpful assistant.', question))
print('\n' + '=' * 50 + '\n')

=== 일반 assistant ===
하루에 커피 4잔을 마시는 것은 일반적으로 건강한 성인에게는 괜찮다고 여겨집니다. 많은 연구에 따르면, 하루 400mg 이하의 카페인은 대부분의 사람들에게 안전하다고 알려져 있습니다. 이는 대략 커피 4잔에 해당합니다. 

하지만 개인의 카페인 민감도는 다를 수 있으며, 일부 사람들은 카페인에 더 민감하여 불안, 불면증, 심장 두근거림 등의 증상을 경험할 수 있습니다. 또한, 임신 중이거나 특정 건강 문제가 있는 경우에는 카페인 섭취를 제한하는 것이 좋습니다.

따라서, 본인의 몸 상태와 반응을 고려하여 적절한 양을 조절하는 것이 중요합니다. 만약 커피를 마신 후 불편한 증상이 있다면 섭취량을 줄이는 것이 좋습니다.




In [8]:
print('=== 영양사 역할 ===')
print(ask(
    'You are a registered dietitian. Answer in Korean with caffeine mg estimate and health advice.',
    question,
))

=== 영양사 역할 ===
하루에 커피 4잔을 마시는 것은 일반적으로 건강한 성인에게는 괜찮다고 여겨집니다. 커피 한 잔에는 대략 95mg의 카페인이 포함되어 있으므로, 4잔을 마시면 약 380mg의 카페인을 섭취하게 됩니다. 

미국 식품의약국(FDA)에서는 건강한 성인이 하루에 최대 400mg의 카페인을 섭취하는 것이 안전하다고 권장하고 있습니다. 하지만 개인의 카페인 민감도는 다를 수 있으므로, 카페인에 민감한 사람은 불안감, 불면증, 심장 두근거림 등의 증상이 나타날 수 있습니다.

따라서 커피를 마시는 것이 괜찮지만, 개인의 상태와 반응을 잘 살펴보는 것이 중요합니다. 또한, 수분 섭취와 균형 잡힌 식사를 함께 고려하여 건강을 유지하는 것이 좋습니다.


In [9]:
roles = [
    "당신은 친절한 고객 서비스 상담원입니다.",
    "당신은 전문적인 기술 문서 작성자입니다.",
    "당신은 창의적인 소설가입니다."
]

question = "인공지능에 대해 설명해주세요."

for role in roles:
    user_prompt = f"{role}\n\n질문: {question}\n\n답변:"
    print(f"=== {role} ===")
    print(f"답변: {ask('Answer in Korean.', user_prompt, temperature=0.8)}")
    print()

=== 당신은 친절한 고객 서비스 상담원입니다. ===
답변: 안녕하세요! 인공지능(AI)은 컴퓨터 시스템이나 소프트웨어가 인간의 지능을 모방하여 학습, 추론, 문제 해결, 이해 및 언어 처리 등의 작업을 수행할 수 있도록 하는 기술입니다. 

인공지능은 크게 두 가지 유형으로 나눌 수 있습니다: 

1. **좁은 인공지능(Weak AI)**: 특정 작업을 수행하도록 설계된 인공지능으로, 예를 들어 음성 인식 시스템, 추천 알고리즘 및 자율주행차의 AI 등이 있습니다.

2. **강한 인공지능(Strong AI)**: 인간 수준의 지능을 갖춘 인공지능으로, 감정, 이해 및 인지 능력을 갖춘 것으로 가정합니다. 현재로서는 이론적인 개념에 가까운 상태입니다.

인공지능의 발전은 다양한 분야에서 혁신을 가져오고 있으며, 의료, 금융, 제조업, 서비스업 등에서 활용되고 있습니다. 궁금한 점이 더 있으시면 언제든지 물어보세요!

=== 당신은 전문적인 기술 문서 작성자입니다. ===
답변: 인공지능(AI, Artificial Intelligence)은 기계나 소프트웨어가 인간의 인지 기능을 모방하여 학습, 추론, 문제 해결, 의사 결정 등의 작업을 수행할 수 있도록 하는 기술 분야입니다. 인공지능은 일반적으로 두 가지 주요 범주로 나눌 수 있습니다.

1. **약한 인공지능(Weak AI)**: 특정 작업을 수행하도록 설계된 인공지능으로, 예를 들어 챗봇, 음성 인식 소프트웨어, 추천 시스템 등이 있습니다. 이러한 시스템은 인간처럼 사고하고 판단할 수 있는 능력이 없으며, 주어진 범위 내에서만 작동합니다.

2. **강한 인공지능(Strong AI)**: 인간과 유사한 수준의 인지를 가지고 모든 지적 작업을 수행할 수 있는 인공지능입니다. 현재까지는 이론적인 개념에 불과하며, 실제 구현은 이루어지지 않았습니다.

인공지능은 머신러닝(기계 학습)과 딥러닝(심층 학습) 기술을 통해 발전해왔습니다. 머신러닝은 데이터를 통해 패턴을 학습하고 예측을 할 수 있도록 하는 알고리즘

---
## 3. 출력 형식 지정 

In [10]:
topic = '재택근무의 장단점 3가지'
system_ko = 'Answer in Korean.'

print('=== Markdown ===')
print(ask(system_ko, topic + '\n형식: markdown bullet point'))
print('\n' + '=' * 50 + '\n')

=== Markdown ===
### 재택근무의 장단점

#### 장점
- **유연한 근무 시간**: 개인의 일정에 맞춰 근무 시간을 조정할 수 있어 일과 삶의 균형을 맞추기 용이함.
- **교통비 및 시간 절약**: 출퇴근 시간이 없으므로 교통비와 시간을 절약할 수 있음.
- **편안한 근무 환경**: 자신이 편안하게 느끼는 환경에서 일할 수 있어 집중력이 향상될 수 있음.

#### 단점
- **사회적 고립감**: 동료와의 직접적인 소통이 줄어들어 고립감을 느낄 수 있음.
- **업무와 개인 생활의 경계 모호**: 집에서 일하다 보면 업무와 개인 생활의 경계가 흐려질 수 있음.
- **자기 관리 필요**: 스스로 동기 부여를 해야 하며, 자율성이 요구되어 자기 관리가 어려울 수 있음.




In [11]:
print('=== Table ===')
print(ask(
    system_ko,
    topic + '\n형식: markdown 표 (항목 | 장점 | 단점)만 출력',
))

=== Table ===
| 항목       | 장점                          | 단점                          |
|------------|-------------------------------|-------------------------------|
| 유연성     | 근무 시간과 장소의 유연성    | 일과 개인 생활의 경계 모호함 |
| 비용 절감  | 교통비 및 식비 절감           | 가정에서의 추가 비용 발생    |
| 생산성     | 집중할 수 있는 환경 조성      | 외부 자극으로 인한 방해 가능  |


---
### 출력 형식 (표 / JSON)

In [12]:
import json

topic = '식각(Etch) 공정에서 수율에 영향 주는 요인 3가지'
system_ko = 'Answer in Korean.'

print('=== Table ===')
print(ask(
    system_ko,
    topic + '\n형식: markdown 표 (요인 | 설명 | 대응)만 출력',
))

=== Table ===
| 요인         | 설명                                 | 대응                     |
|--------------|--------------------------------------|--------------------------|
| 화학물질     | 식각에 사용되는 화학물질의 농도와 조성 | 적절한 화학물질 선택 및 농도 조절 |
| 온도         | 공정 중 온도가 수율에 미치는 영향   | 온도 모니터링 및 조절    |
| 시간         | 식각 시간의 길이에 따른 결과 차이   | 최적의 식각 시간 설정    |


In [13]:
print('=== JSON ===')
result = ask(
    'Output ONLY valid JSON. No markdown.',
    topic + '\n{"factors": [{"name": "", "action": ""}]} 형식으로',
)
print(result)
try:
    print('\n파싱 성공:', list(json.loads(result).keys()))
except json.JSONDecodeError:
    print('\n파싱 실패 — ONLY valid JSON 강조 필요')

=== JSON ===
{
  "factors": [
    {
      "name": "공정 조건",
      "action": "온도, 압력, 화학물질 농도 등을 최적화하여 수율을 향상시킨다."
    },
    {
      "name": "마스크 품질",
      "action": "마스크의 정밀도와 결함 유무가 식각 결과에 영향을 미친다."
    },
    {
      "name": "기판 재료",
      "action": "기판의 특성과 표면 상태가 식각 공정의 효율성에 영향을 준다."
    }
  ]
}

파싱 성공: ['factors']


---
## 4. Chain-of-Thought (CoT)

CoT는 모델이 단계별로 추론 과정을 보여주도록 하는 기법.

In [14]:
## zero-shot 예시

problem = '''
A팀 10명, B팀 15명, C팀 5명.
전체 25명 중 40% 이상이 A팀이면 A팀에 추가 인원 배치.
지금 추가 배치가 필요한가?
'''
system = 'You are a helpful assistant. Answer in Korean.'
print('=== CoT 없음 ===')
print(ask(system, problem))

=== CoT 없음 ===
전체 25명 중 40%는 10명입니다. 현재 A팀은 10명으로, 전체 인원의 40%에 해당합니다. 따라서 A팀에 추가 배치가 필요하지 않습니다. A팀의 인원이 40% 이상이 되려면 11명 이상이어야 합니다. 현재 A팀은 40%에 해당하므로 추가 배치가 필요하지 않습니다.


In [15]:
print('=== CoT 적용 ===')
print(ask(
    system + ' Show step-by-step reasoning before final answer.',
    problem + '\n\n단계별로 생각해 봅시다.',
))

=== CoT 적용 ===
먼저, 전체 인원 수와 A팀의 인원 수를 확인해 보겠습니다.

1. **전체 인원 수**: A팀 10명 + B팀 15명 + C팀 5명 = 30명
2. **A팀 인원 수**: 10명

이제 A팀의 인원이 전체 인원 중 몇 퍼센트인지 계산해 보겠습니다.

3. **A팀 비율 계산**:
   \[
   \text{A팀 비율} = \left( \frac{\text{A팀 인원}}{\text{전체 인원}} \right) \times 100 = \left( \frac{10}{30} \right) \times 100 = 33.33\%
   \]

4. **40% 이상인지 확인**: A팀의 비율인 33.33%는 40%보다 작습니다.

따라서, A팀에 추가 인원 배치가 필요하지 않습니다.

최종 결론: **추가 배치가 필요하지 않습니다.**


In [16]:
problem = """다음 처럼 문제를 답하시오.

문제: 한 상점에서 연필 5자루와 지우개 3개를 샀습니다. 연필은 개당 500원, 지우개는 개당 300원입니다. 총 금액은 얼마인가요?

답: ?원"""

system = 'You are a helpful assistant. Answer in Korean.'

print(ask(system, problem))

답: 3,600원


In [17]:
problem = """다음 처럼 문제를 답하시오.

문제: 한 상점에서 연필 5자루와 지우개 3개를 샀습니다. 연필은 개당 500원, 지우개는 개당 300원입니다. 총 금액은 얼마인가요?

천천히 단계별로 생각해봅시다.

답: ?원"""

system = 'You are a helpful assistant. Answer in Korean.'

print(ask(system, problem))

문제에서 주어진 정보를 바탕으로 단계별로 계산해보겠습니다.

1. 연필의 가격: 500원
2. 지우개의 가격: 300원
3. 연필의 개수: 5자루
4. 지우개의 개수: 3개

이제 각각의 총 금액을 계산해보겠습니다.

1. 연필의 총 금액:
   - 5자루 × 500원 = 2500원

2. 지우개의 총 금액:
   - 3개 × 300원 = 900원

이제 두 금액을 합산합니다.

총 금액 = 연필의 총 금액 + 지우개의 총 금액
총 금액 = 2500원 + 900원 = 3400원

따라서, 총 금액은 3400원입니다.

답: 3400원


In [34]:
problem = """다음 처럼 문제를 답하시오.

문제: 한 상점에서 사과 3개와 배 2개를 샀습니다. 사과는 개당 1000원, 배는 개당 1500원입니다. 총 금액은 얼마인가요?

답: 6000원

---

문제: 한 상점에서 귤 1개와 바나나 12개를 샀습니다. 사과는 개당 1000원, 배는 개당 200원입니다. 총 금액은 얼마인가요?

답: 3400원

---


문제: 한 상점에서 연필 5자루와 지우개 3개를 샀습니다. 연필은 개당 500원, 지우개는 개당 300원입니다. 총 금액은 얼마인가요?

답: """

system = 'You are a helpful assistant. Answer in Korean.'

print(ask(system, problem))

답: 3900원


In [25]:
## one-shot 예시

problem = """다음 문제를 단계별로 풀어보세요.

문제: 한 상점에서 사과 3개와 배 2개를 샀습니다. 사과는 개당 1000원, 배는 개당 1500원입니다. 총 금액은 얼마인가요?

단계별 풀이:
1. 사과 3개의 가격: 3 × 1000 = 3000원
2. 배 2개의 가격: 2 × 1500 = 3000원
3. 총 금액: 3000 + 3000 = 6000원
답: 6000원

---

문제: 한 상점에서 연필 5자루와 지우개 3개를 샀습니다. 연필은 개당 500원, 지우개는 개당 300원입니다. 총 금액은 얼마인가요?

단계별 풀이:"""

system = 'You are a helpful assistant. Answer in Korean.'

print(ask(system, problem))

단계별 풀이:

1. 연필 5자루의 가격: 5 × 500 = 2500원
2. 지우개 3개의 가격: 3 × 300 = 900원
3. 총 금액: 2500 + 900 = 3400원

답: 3400원


---
## 5. System + User 프롬프트 

In [35]:
system_prompt = '''
너는 스타트업 PM의 일정 관리 AI다.
규칙: 결론 먼저, bullet 3개, 한국어
'''

user_prompt = '''
이번 주 마감: 월-기획안, 수-코드리뷰, 금-발표.
목요일에 하루 종일 회의가 잡혔어. 우선순위 조정해줘.
'''

print(ask(system_prompt, user_prompt))

결론: 목요일 회의로 인해 월요일과 수요일의 작업을 조정해야 합니다.

- 월요일: 기획안 작업을 완료하고, 수요일에 코드리뷰를 진행합니다.
- 수요일: 코드리뷰 후, 필요한 경우 목요일 회의 준비를 위해 추가 작업을 합니다.
- 금요일: 발표 준비를 최종 점검하고 발표를 진행합니다.


In [36]:
system_md = (
    "너는 백화점 MD 업무를 보조하는 AI다. "
    "답변은 반드시 표 형태로 작성하고, "
    "결론을 먼저 제시한 뒤 "
    "리스크와 가정을 명시해야 한다."
)

user_md = "이번 주 VIP 이탈 상위 200명에 대한 액션 플랜을 작성해줘."

answer = ask(system_md, user_md, temperature=0.7)
print(answer)

| 결론                                       |
|--------------------------------------------|
| VIP 이탈 방지를 위한 맞춤형 액션 플랜 수립  |

| 리스크                                      | 가정                                       |
|--------------------------------------------|-------------------------------------------|
| 1. 고객 이탈 원인 분석 실패로 인한 비효율적인 대응 | 1. VIP 고객의 이탈은 특정 요인에 의해 발생한다.   |
| 2. 할인 및 프로모션이 고객을 돌아오게 하지 못할 가능성 | 2. 고객들은 개인화된 서비스 및 혜택에 반응한다.   |
| 3. VIP 고객의 관심과 니즈 변화에 대한 반응 속도 느림 | 3. 고객 데이터 분석을 통해 인사이트를 도출할 수 있다. |

| 액션 플랜                                   | 세부 내용                                    |
|--------------------------------------------|--------------------------------------------|
| 1. 데이터 분석 및 리포트 작성                   | VIP 고객 이탈 원인 및 패턴 분석, 이탈 고객 목록 작성 |
| 2. 맞춤형 커뮤니케이션 전략 수립               | 개인화된 이메일 및 문자 발송, 고객의 선호에 맞춘 메시지 전달 |
| 3. 독점 혜택 제공                                 | VIP 전용 할인 쿠폰, 특별 이벤트 초대 등 제공            |
| 4. 피드백 수집 및 분석                         | 이탈 고객과의 인터뷰, 설문 조

---
## 6. Self-check Prompting

In [37]:
check_prompt = f"""아래는 네가 작성한 'VIP 이탈 상위 200명 액션 플랜' 초안이다.

[초안]
{answer}

이제 아래 체크리스트로 점검하고, 필요하면 수정본을 작성하라.

체크리스트:
1) 논리적 모순/비약이 있는가?
2) 실행 불가능하거나 모호한 액션이 있는가? (담당/기한/방법이 불명확한지)
3) 과도한 가정이 있는가?
4) 표 형식이 지켜졌는가? (결론 먼저, 리스크/가정 포함)

출력 규칙:
- 먼저 "점검 결과"를 bullet로 간단히 쓰고
- 그 다음 "수정본"을 작성하라
- 수정본은 반드시 표 형태 + 결론 먼저 + 리스크/가정 명시
"""

final_answer = ask(system_md, check_prompt, temperature=0.7)
print('\n--- Self-check 후 최종 답변(수정본) ---')
print(final_answer)


--- Self-check 후 최종 답변(수정본) ---
### 점검 결과
- 논리적 모순/비약 없음
- 실행 가능성이 있는 액션이나 담당/기한/방법이 명확하지 않음
- 과도한 가정 없음
- 표 형식이 지켜짐

### 수정본
| 결론                                       |
|--------------------------------------------|
| VIP 이탈 방지를 위한 맞춤형 액션 플랜 수립  |

| 리스크                                      | 가정                                       |
|--------------------------------------------|-------------------------------------------|
| 1. 고객 이탈 원인 분석 실패로 인한 비효율적인 대응 | 1. VIP 고객의 이탈은 특정 요인에 의해 발생한다.   |
| 2. 할인 및 프로모션이 고객을 돌아오게 하지 못할 가능성 | 2. 고객들은 개인화된 서비스 및 혜택에 반응한다.   |
| 3. VIP 고객의 관심과 니즈 변화에 대한 반응 속도 느림 | 3. 고객 데이터 분석을 통해 인사이트를 도출할 수 있다. |
| 4. 리워드 프로그램의 효과성 부족               | 4. 고객은 리워드 프로그램에 참여할 의사가 있다.     |

| 액션 플랜                                   | 세부 내용                                    | 담당자       | 기한         |
|--------------------------------------------|--------------------------------------------|------------|-------------|
| 1. 데이터 분석 및 리포트 작성                   | VIP 고객 이탈 

---
## 7. Constraint Prompting

In [38]:
system_prompt = (
    "너는 백화점 MD 업무를 보조하는 AI다. "
    "답변은 반드시 표 형태로 작성하고, "
    "결론을 먼저 제시한 뒤, "
    "리스크와 가정을 명시해야 한다."
)

constraints = """조건(반드시 준수):
- 추가 예산은 최대 1억 원 이내
- 인력 증원 불가 (기존 인력 내에서 운영)
- 2주 이내 실행 가능한 액션만 제시
- 고객에게 직접 연락하는 액션은 과도한 빈도를 피하고(1회 이내),
  개인정보/컴플라이언스 리스크를 고려할 것
"""

user_prompt = f"""이번 주 VIP 이탈 상위 200명에 대한 액션 플랜을 작성해줘.

{constraints}

출력 형식:
- 결론(한 줄) 먼저
- 그 다음 표로 정리 (컬럼 예: 타깃/액션/채널/우선순위/기간/기대효과/담당)
- 마지막에 리스크/가정 명시
"""

constrained_answer = ask(system_prompt, user_prompt, temperature=0.7)
print('\n--- 조건 기반 최종 답변(답변만) ---')
print(constrained_answer)


--- 조건 기반 최종 답변(답변만) ---
결론: VIP 이탈 상위 200명에 대해 맞춤형 프로모션과 개인화된 경험을 제공하여 이탈 방지 및 재방문 유도.

| 타깃           | 액션                         | 채널         | 우선순위 | 기간      | 기대효과                     | 담당      |
|----------------|------------------------------|--------------|----------|-----------|------------------------------|-----------|
| VIP 상위 200명 | 맞춤형 할인 쿠폰 발송       | 이메일       | 높음     | 1주 이내  | 재방문 유도 및 고객 충성도 증가 | 마케팅팀  |
| VIP 상위 200명 | 특별 이벤트 초대 (온라인)   | SNS/이메일   | 중간     | 2주 이내  | 고객 관계 강화 및 브랜드 이미지 개선 | 이벤트팀  |
| VIP 상위 200명 | 개인화된 쇼핑 리포트 제공   | 이메일       | 높음     | 1주 이내  | 고객의 쇼핑 패턴 이해 및 재구매 유도 | 데이터팀  |
| VIP 상위 200명 | 고객 피드백 요청            | 이메일       | 낮음     | 2주 이내  | 고객의 니즈 파악 및 서비스 개선 | 고객지원팀 |

### 리스크 및 가정
- **리스크**: 고객의 개인정보 보호 및 컴플라이언스 위반 가능성, 할인 쿠폰 남용으로 인한 수익 감소.
- **가정**: 고객들이 제공된 혜택에 긍정적으로 반응할 것, 기존 인력이 액션 플랜 실행에 충분한 역량을 가지고 있을 것.
